In [1]:
# ============================================================
# Step 1 — Environment, paths, device, and output directories
# New notebook:
# train_point_diffusion_progressive_distillation.ipynb
# ============================================================

from pathlib import Path
import sys
import platform
import random

import numpy as np
import torch

print("=" * 88)
print("Step 1 — Distillation environment and path check")
print("=" * 88)

# ------------------------------------------------------------
# 1. Reproducibility
# ------------------------------------------------------------
GLOBAL_SEED = 42

random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)
torch.manual_seed(GLOBAL_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(GLOBAL_SEED)

# ------------------------------------------------------------
# 2. Main project paths
# ------------------------------------------------------------
PROJECT_ROOT = Path("/home/nuoxu9/PIDIF")

DEEPO_NET_CODE_DIR = PROJECT_ROOT / "DeepONet"

DATASET_H5 = (
    PROJECT_ROOT
    / "channel_diffusion_dataset"
    / "deeponet_style_dataset"
    / "channel_deeponet_style_pressure_u_v_controlpoints.h5"
)

TEACHER_CKPT = (
    PROJECT_ROOT
    / "channel_diffusion_dataset"
    / "point_diffusion_deeponet_style_puv_baseline_long"
    / "checkpoints"
    / "point_diffusion_baseline_long_best.pt"
)

DISTILLATION_ROOT = (
    PROJECT_ROOT
    / "channel_diffusion_dataset"
    / "point_diffusion_progressive_distillation"
)

CHECKPOINT_DIR = DISTILLATION_ROOT / "checkpoints"
LOG_DIR = DISTILLATION_ROOT / "logs"
EVAL_DIR = DISTILLATION_ROOT / "evaluation"
CACHE_DIR = DISTILLATION_ROOT / "teacher_cache"

for directory in [
    DISTILLATION_ROOT,
    CHECKPOINT_DIR,
    LOG_DIR,
    EVAL_DIR,
    CACHE_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 3. Add senior DeepONet code to Python path
# ------------------------------------------------------------
if str(DEEPO_NET_CODE_DIR) not in sys.path:
    sys.path.insert(0, str(DEEPO_NET_CODE_DIR))

# ------------------------------------------------------------
# 4. Device selection
#
# We continue using CUDA device 1, consistent with the
# baseline and step-ablation notebook.
# ------------------------------------------------------------
REQUESTED_GPU_ID = 1

if torch.cuda.is_available():
    if torch.cuda.device_count() <= REQUESTED_GPU_ID:
        raise RuntimeError(
            f"Requested cuda:{REQUESTED_GPU_ID}, but only "
            f"{torch.cuda.device_count()} CUDA device(s) are visible."
        )

    DEVICE = torch.device(f"cuda:{REQUESTED_GPU_ID}")
    torch.cuda.set_device(DEVICE)

else:
    DEVICE = torch.device("cpu")

# ------------------------------------------------------------
# 5. Basic configuration for the first distillation experiment
# ------------------------------------------------------------
DISTILLATION_CONFIG = {
    "global_seed": GLOBAL_SEED,
    "teacher_sampling_steps": 50,
    "student_sampling_steps": 5,
    "diffusion_training_steps": 1000,
    "target_fields": ["pressure", "u", "v"],
    "teacher_forcing_interfaces": True,
    "use_physics_loss": False,
    "use_interface_corruption": False,
}

# ------------------------------------------------------------
# 6. Path checks
# ------------------------------------------------------------
required_paths = {
    "project_root": PROJECT_ROOT,
    "deeponet_code_dir": DEEPO_NET_CODE_DIR,
    "dataset_h5": DATASET_H5,
    "teacher_checkpoint": TEACHER_CKPT,
}

missing_required_paths = []

print("\nRequired paths")
for name, path in required_paths.items():
    exists = path.exists()

    print(
        f"{name:22s}: {exists} | {path}"
    )

    if not exists:
        missing_required_paths.append(
            (name, path)
        )

# ------------------------------------------------------------
# 7. Environment summary
# ------------------------------------------------------------
print("\nEnvironment")
print("Python version      :", platform.python_version())
print("PyTorch version     :", torch.__version__)
print("NumPy version       :", np.__version__)
print("CUDA available      :", torch.cuda.is_available())
print("CUDA device count   :", torch.cuda.device_count())
print("selected device     :", DEVICE)

if DEVICE.type == "cuda":
    device_index = DEVICE.index

    print(
        "GPU name            :",
        torch.cuda.get_device_name(device_index),
    )

    print(
        "GPU capability      :",
        torch.cuda.get_device_capability(device_index),
    )

    print(
        "allocated memory GB :",
        torch.cuda.memory_allocated(device_index) / 1024**3,
    )

    print(
        "reserved memory GB  :",
        torch.cuda.memory_reserved(device_index) / 1024**3,
    )

# ------------------------------------------------------------
# 8. Output directories
# ------------------------------------------------------------
print("\nOutput directories")
print("distillation root :", DISTILLATION_ROOT)
print("checkpoints       :", CHECKPOINT_DIR)
print("logs              :", LOG_DIR)
print("evaluation        :", EVAL_DIR)
print("teacher cache     :", CACHE_DIR)

# ------------------------------------------------------------
# 9. Experiment configuration
# ------------------------------------------------------------
print("\nInitial distillation configuration")
for key, value in DISTILLATION_CONFIG.items():
    print(f"{key:28s}: {value}")

# ------------------------------------------------------------
# 10. Final validation
# ------------------------------------------------------------
print("\n" + "=" * 88)

if len(missing_required_paths) == 0:
    print("[READY] Distillation environment initialized successfully.")
    print(
        "Next step: load the existing H5 dataset and recover "
        "the original train/validation/test split."
    )
else:
    print("[ERROR] Required paths are missing:")

    for name, path in missing_required_paths:
        print(f"  - {name}: {path}")

    raise FileNotFoundError(
        "One or more required distillation inputs are missing."
    )

print("=" * 88)

Step 1 — Distillation environment and path check

Required paths
project_root          : True | /home/nuoxu9/PIDIF
deeponet_code_dir     : True | /home/nuoxu9/PIDIF/DeepONet
dataset_h5            : True | /home/nuoxu9/PIDIF/channel_diffusion_dataset/deeponet_style_dataset/channel_deeponet_style_pressure_u_v_controlpoints.h5
teacher_checkpoint    : True | /home/nuoxu9/PIDIF/channel_diffusion_dataset/point_diffusion_deeponet_style_puv_baseline_long/checkpoints/point_diffusion_baseline_long_best.pt

Environment
Python version      : 3.11.14
PyTorch version     : 2.10.0+cu128
NumPy version       : 2.4.2
CUDA available      : True
CUDA device count   : 4
selected device     : cuda:1
GPU name            : NVIDIA RTX A6000
GPU capability      : (8, 6)
allocated memory GB : 0.0
reserved memory GB  : 0.0

Output directories
distillation root : /home/nuoxu9/PIDIF/channel_diffusion_dataset/point_diffusion_progressive_distillation
checkpoints       : /home/nuoxu9/PIDIF/channel_diffusion_dataset/po

In [2]:
# ============================================================
# Step 2 — Load dataset and restore the exact baseline split
#
# Sources of truth:
#   Dataset content  -> existing H5 file
#   Split/normalizer -> frozen teacher checkpoint
#
# No training is performed.
# ============================================================

from collections import Counter

import numpy as np
import torch

from deeponet_fluent_dataset import (
    load_deeponet_dataset_h5,
)

from fluent_deeponet import (
    FeatureNormalizer,
)

print("=" * 92)
print("Step 2 — Load dataset and restore baseline split")
print("=" * 92)

# ------------------------------------------------------------
# 1. Load existing H5 dataset
# ------------------------------------------------------------
print("\nLoading dataset:")
print(DATASET_H5)

data = load_deeponet_dataset_h5(DATASET_H5)

samples = list(data["samples"])
metadata = list(data["metadata"])

branch_channel_names = list(
    data["branch_channel_names"]
)

trunk_channel_names = list(
    data["trunk_channel_names"]
)

output_channel_names = list(
    data["output_channel_names"]
)

n_samples = len(samples)

if len(metadata) != n_samples:
    raise RuntimeError(
        f"Sample/metadata mismatch: "
        f"{n_samples} vs {len(metadata)}"
    )

print("\nDataset loaded")
print("samples              :", n_samples)
print("metadata             :", len(metadata))
print("branch channels      :", branch_channel_names)
print("trunk channels       :", trunk_channel_names)
print("output channels      :", output_channel_names)
print("n interface points   :", data["n_interface_points"])
print("n boundary points    :", data["n_boundary_points"])
print("horizontal interface :", data["horizontal_interface"])

# ------------------------------------------------------------
# 2. Inspect representative tensor shapes
# ------------------------------------------------------------
sample0 = samples[0]

print("\nFirst sample")
print("branch:", sample0["branch"].shape, sample0["branch"].dtype)
print("query :", sample0["query"].shape, sample0["query"].dtype)
print("target:", sample0["target"].shape, sample0["target"].dtype)

BRANCH_DIM = int(sample0["branch"].shape[-1])
QUERY_DIM = int(sample0["query"].shape[-1])
TARGET_DIM = int(sample0["target"].shape[-1])

# ------------------------------------------------------------
# 3. Load teacher checkpoint metadata
# ------------------------------------------------------------
print("\nLoading teacher checkpoint metadata:")
print(TEACHER_CKPT)

teacher_checkpoint = torch.load(
    TEACHER_CKPT,
    map_location="cpu",
    weights_only=False,
)

print("\nCheckpoint keys")
print(list(teacher_checkpoint.keys()))

required_checkpoint_keys = [
    "model_state_dict",
    "model_config",
    "diffusion_config",
    "y_normalizer",
    "local_aspect_mean",
    "local_aspect_std",
    "train_cases",
    "val_cases",
    "test_cases",
]

missing_checkpoint_keys = [
    key
    for key in required_checkpoint_keys
    if key not in teacher_checkpoint
]

if missing_checkpoint_keys:
    raise KeyError(
        "Teacher checkpoint is missing required keys: "
        f"{missing_checkpoint_keys}"
    )

# ------------------------------------------------------------
# 4. Restore exact case split
# ------------------------------------------------------------
train_cases = [
    str(case_id)
    for case_id in teacher_checkpoint["train_cases"]
]

val_cases = [
    str(case_id)
    for case_id in teacher_checkpoint["val_cases"]
]

test_cases = [
    str(case_id)
    for case_id in teacher_checkpoint["test_cases"]
]

train_case_set = set(train_cases)
val_case_set = set(val_cases)
test_case_set = set(test_cases)

# Verify no case overlap
if train_case_set & val_case_set:
    raise RuntimeError("Train and validation cases overlap.")

if train_case_set & test_case_set:
    raise RuntimeError("Train and test cases overlap.")

if val_case_set & test_case_set:
    raise RuntimeError("Validation and test cases overlap.")

# Map each sample to its case split
train_idx = np.asarray(
    [
        i
        for i, row in enumerate(metadata)
        if str(row["case_id"]) in train_case_set
    ],
    dtype=np.int64,
)

val_idx = np.asarray(
    [
        i
        for i, row in enumerate(metadata)
        if str(row["case_id"]) in val_case_set
    ],
    dtype=np.int64,
)

test_idx = np.asarray(
    [
        i
        for i, row in enumerate(metadata)
        if str(row["case_id"]) in test_case_set
    ],
    dtype=np.int64,
)

all_split_indices = np.concatenate(
    [train_idx, val_idx, test_idx]
)

if len(np.unique(all_split_indices)) != len(all_split_indices):
    raise RuntimeError(
        "Duplicate sample indices exist across splits."
    )

if len(all_split_indices) != n_samples:
    dataset_case_ids = {
        str(row["case_id"])
        for row in metadata
    }

    checkpoint_case_ids = (
        train_case_set
        | val_case_set
        | test_case_set
    )

    missing_from_checkpoint = sorted(
        dataset_case_ids - checkpoint_case_ids
    )

    missing_from_dataset = sorted(
        checkpoint_case_ids - dataset_case_ids
    )

    raise RuntimeError(
        "Checkpoint split does not cover the complete dataset.\n"
        f"covered samples: {len(all_split_indices)} / {n_samples}\n"
        f"dataset-only cases: {missing_from_checkpoint}\n"
        f"checkpoint-only cases: {missing_from_dataset}"
    )

# ------------------------------------------------------------
# 5. Restore exact target normalizer
# ------------------------------------------------------------
y_normalizer = FeatureNormalizer.from_state_dict(
    teacher_checkpoint["y_normalizer"]
)

local_aspect_mean = float(
    teacher_checkpoint["local_aspect_mean"]
)

local_aspect_std = float(
    teacher_checkpoint["local_aspect_std"]
)

print("\nNormalizer")
print(
    "target mean:",
    y_normalizer.mean.detach().cpu().numpy(),
)

print(
    "target std :",
    y_normalizer.std.detach().cpu().numpy(),
)

print("local aspect mean:", local_aspect_mean)
print("local aspect std :", local_aspect_std)

# Keep the normalizer on CPU for dataset preprocessing.
# It can be copied to DEVICE during training/evaluation.
y_normalizer_cpu = FeatureNormalizer.from_state_dict(
    teacher_checkpoint["y_normalizer"]
)

# ------------------------------------------------------------
# 6. Verify split structure
# ------------------------------------------------------------
def summarize_split(name, indices, expected_cases):
    split_case_ids = [
        str(metadata[int(i)]["case_id"])
        for i in indices
    ]

    split_subdomain_ids = [
        int(metadata[int(i)]["subdomain_id"])
        for i in indices
    ]

    case_counts = Counter(split_case_ids)

    print(f"\n{name}")
    print("cases       :", len(set(split_case_ids)))
    print("samples     :", len(indices))
    print(
        "samples/case:",
        sorted(set(case_counts.values())),
    )
    print(
        "subdomain range:",
        min(split_subdomain_ids),
        "to",
        max(split_subdomain_ids),
    )

    if set(split_case_ids) != set(expected_cases):
        raise RuntimeError(
            f"{name} case IDs do not match checkpoint."
        )

    return case_counts


train_case_counts = summarize_split(
    "Train split",
    train_idx,
    train_cases,
)

val_case_counts = summarize_split(
    "Validation split",
    val_idx,
    val_cases,
)

test_case_counts = summarize_split(
    "Test split",
    test_idx,
    test_cases,
)

# ------------------------------------------------------------
# 7. Verify dataset/checkpoint compatibility
# ------------------------------------------------------------
checkpoint_output_names = list(
    teacher_checkpoint["output_channel_names"]
)

checkpoint_branch_names = list(
    teacher_checkpoint["branch_channel_names"]
)

checkpoint_trunk_names = list(
    teacher_checkpoint["trunk_channel_names"]
)

if output_channel_names != checkpoint_output_names:
    raise RuntimeError(
        "Dataset/checkpoint output channel mismatch:\n"
        f"dataset    : {output_channel_names}\n"
        f"checkpoint : {checkpoint_output_names}"
    )

if branch_channel_names != checkpoint_branch_names:
    raise RuntimeError(
        "Dataset/checkpoint branch channel mismatch:\n"
        f"dataset    : {branch_channel_names}\n"
        f"checkpoint : {checkpoint_branch_names}"
    )

if trunk_channel_names != checkpoint_trunk_names:
    raise RuntimeError(
        "Dataset/checkpoint trunk channel mismatch:\n"
        f"dataset    : {trunk_channel_names}\n"
        f"checkpoint : {checkpoint_trunk_names}"
    )

# ------------------------------------------------------------
# 8. Final summary
# ------------------------------------------------------------
print("\n" + "=" * 92)
print("Restored configuration")
print("=" * 92)

print("BRANCH_DIM :", BRANCH_DIM)
print("QUERY_DIM  :", QUERY_DIM)
print("TARGET_DIM :", TARGET_DIM)

print("\nCase split")
print("train cases:", len(train_cases))
print("val cases  :", len(val_cases))
print("test cases :", len(test_cases))

print("\nSample split")
print("train_idx:", train_idx.shape)
print("val_idx  :", val_idx.shape)
print("test_idx :", test_idx.shape)

print("\nTeacher checkpoint")
print("epoch        :", teacher_checkpoint.get("epoch"))
print("best val loss:", teacher_checkpoint.get("best_val_loss"))
print("dataset H5   :", teacher_checkpoint.get("dataset_h5"))

print("\n" + "=" * 92)
print("[READY] Dataset, exact split, and normalizers restored.")
print(
    "Next step: construct train/validation/test datasets and "
    "data loaders using the same preprocessing as the teacher."
)
print("=" * 92)

Step 2 — Load dataset and restore baseline split

Loading dataset:
/home/nuoxu9/PIDIF/channel_diffusion_dataset/deeponet_style_dataset/channel_deeponet_style_pressure_u_v_controlpoints.h5

Dataset loaded
samples              : 2000
metadata             : 2000
branch channels      : ['x_local', 'y_local', 'wall_mask', 'interface_mask', 'boundary_pressure', 'boundary_u', 'boundary_v', 'known_pressure', 'known_u', 'known_v', 'local_aspect_ratio']
trunk channels       : ['x_local', 'y_local']
output channels      : ['pressure', 'u', 'v']
n interface points   : 256
n boundary points    : 256
horizontal interface : False

First sample
branch: (1024, 11) float32
query : (28600, 2) float32
target: (28600, 3) float32

Loading teacher checkpoint metadata:
/home/nuoxu9/PIDIF/channel_diffusion_dataset/point_diffusion_deeponet_style_puv_baseline_long/checkpoints/point_diffusion_baseline_long_best.pt

Checkpoint keys
['epoch', 'model_state_dict', 'optimizer_state_dict', 'scheduler_state_dict', 'trai

In [6]:
# ============================================================
# Step 3 — Rebuild the exact original datasets and dataloaders
#
# Important:
#   - Uses the original DeepONetCellDataset
#   - Uses the original variable-length collate function
#   - N_QUERY_POINTS=8192 means "at most 8192 per sample"
#   - Small subdomains use all available points
#   - No replacement sampling and no artificial duplication
# ============================================================

import numpy as np
import torch

from torch.utils.data import DataLoader

from deeponet_fluent_dataset import (
    DeepONetCellDataset,
    deeponet_cell_collate_fn,
)

print("=" * 92)
print("Step 3 — Restore original variable-length dataloaders")
print("=" * 92)

# ------------------------------------------------------------
# 1. Original training settings
# ------------------------------------------------------------
N_QUERY_POINTS = 8192
BATCH_SIZE = 4
NUM_WORKERS = 0

print("\nDataLoader configuration")
print("maximum query points per sample:", N_QUERY_POINTS)
print("batch size                     :", BATCH_SIZE)
print("num workers                    :", NUM_WORKERS)

# ------------------------------------------------------------
# 2. Build datasets using exact original preprocessing
#
# train:
#   random_query=True
#   large subdomains are resampled each epoch
#
# validation/test:
#   random_query=False
#   deterministic point selection
# ------------------------------------------------------------
train_ds = DeepONetCellDataset(
    samples=data["samples"],
    sample_indices=train_idx,
    n_query_points=N_QUERY_POINTS,
    random_query=True,
    target_y_normalizer=y_normalizer,
    local_aspect_mean=local_aspect_mean,
    local_aspect_std=local_aspect_std,
    branch_channel_names=data["branch_channel_names"],
)

val_ds = DeepONetCellDataset(
    samples=data["samples"],
    sample_indices=val_idx,
    n_query_points=N_QUERY_POINTS,
    random_query=False,
    target_y_normalizer=y_normalizer,
    local_aspect_mean=local_aspect_mean,
    local_aspect_std=local_aspect_std,
    branch_channel_names=data["branch_channel_names"],
)

test_ds = DeepONetCellDataset(
    samples=data["samples"],
    sample_indices=test_idx,
    n_query_points=N_QUERY_POINTS,
    random_query=False,
    target_y_normalizer=y_normalizer,
    local_aspect_mean=local_aspect_mean,
    local_aspect_std=local_aspect_std,
    branch_channel_names=data["branch_channel_names"],
)

print("\nDataset lengths")
print("train:", len(train_ds))
print("val  :", len(val_ds))
print("test :", len(test_ds))

# ------------------------------------------------------------
# 3. Build DataLoaders with original custom collate function
# ------------------------------------------------------------
train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    collate_fn=deeponet_cell_collate_fn,
    pin_memory=(DEVICE.type == "cuda"),
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=deeponet_cell_collate_fn,
    pin_memory=(DEVICE.type == "cuda"),
)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=deeponet_cell_collate_fn,
    pin_memory=(DEVICE.type == "cuda"),
)

print("\nDataLoader lengths")
print("train batches:", len(train_loader))
print("val batches  :", len(val_loader))
print("test batches :", len(test_loader))

# ------------------------------------------------------------
# 4. Helper for unpacking both historical collate formats
# ------------------------------------------------------------
def unpack_deeponet_batch(batch):
    if len(batch) == 6:
        (
            branch,
            query,
            target,
            query_batch_id,
            sample_idx,
            branch_mask,
        ) = batch

    elif len(batch) == 5:
        (
            branch,
            query,
            target,
            query_batch_id,
            sample_idx,
        ) = batch

        branch_mask = None

    else:
        raise RuntimeError(
            f"Unexpected batch length: {len(batch)}"
        )

    return {
        "branch": branch,
        "query": query,
        "target": target,
        "query_batch_id": query_batch_id,
        "sample_idx": sample_idx,
        "branch_mask": branch_mask,
    }


# ------------------------------------------------------------
# 5. Inspect one train batch
# ------------------------------------------------------------
raw_batch = next(iter(train_loader))
batch = unpack_deeponet_batch(raw_batch)

print("\nOne train batch")

for name in [
    "branch",
    "query",
    "target",
    "query_batch_id",
    "sample_idx",
]:
    value = batch[name]

    print(
        f"{name:16s}: "
        f"shape={tuple(value.shape)}, "
        f"dtype={value.dtype}"
    )

if batch["branch_mask"] is not None:
    branch_mask = batch["branch_mask"]

    print(
        f"{'branch_mask':16s}: "
        f"shape={tuple(branch_mask.shape)}, "
        f"dtype={branch_mask.dtype}"
    )

    print(
        "branch_mask valid counts:",
        branch_mask.sum(dim=1),
    )

# ------------------------------------------------------------
# 6. Count query points per sample in this batch
# ------------------------------------------------------------
query_batch_id = batch["query_batch_id"]
batch_size_actual = int(batch["branch"].shape[0])

points_per_sample = torch.bincount(
    query_batch_id,
    minlength=batch_size_actual,
)

print("\nQuery points per sample in this batch")
print(points_per_sample)

print(
    "total concatenated query points:",
    int(points_per_sample.sum()),
)

print(
    "maximum allowed per sample:",
    N_QUERY_POINTS,
)

if torch.any(points_per_sample > N_QUERY_POINTS):
    raise RuntimeError(
        "A sample contains more query points than "
        "N_QUERY_POINTS after dataset sampling."
    )

# ------------------------------------------------------------
# 7. Confirm point-to-branch mapping
# ------------------------------------------------------------
unique_batch_ids = torch.unique(
    query_batch_id
)

print("\nUnique query batch IDs")
print(unique_batch_ids)

expected_ids = torch.arange(
    batch_size_actual,
    dtype=unique_batch_ids.dtype,
)

if not torch.equal(
    unique_batch_ids.cpu(),
    expected_ids.cpu(),
):
    raise RuntimeError(
        "query_batch_id does not cover every branch sample."
    )

# ------------------------------------------------------------
# 8. Numerical checks
# ------------------------------------------------------------
for name in [
    "branch",
    "query",
    "target",
]:
    tensor = batch[name]

    if not torch.isfinite(tensor).all():
        raise RuntimeError(
            f"Non-finite values found in {name}."
        )

print("\nTarget statistics in normalized space")
print(
    "mean:",
    batch["target"].mean(dim=0),
)
print(
    "std :",
    batch["target"].std(dim=0),
)

# ------------------------------------------------------------
# 9. Check deterministic validation data
# ------------------------------------------------------------
val_batch_a = unpack_deeponet_batch(
    next(iter(val_loader))
)

val_batch_b = unpack_deeponet_batch(
    next(iter(val_loader))
)

val_query_same = torch.equal(
    val_batch_a["query"],
    val_batch_b["query"],
)

val_target_same = torch.equal(
    val_batch_a["target"],
    val_batch_b["target"],
)

val_batch_id_same = torch.equal(
    val_batch_a["query_batch_id"],
    val_batch_b["query_batch_id"],
)

print("\nValidation reproducibility")
print("query identical         :", val_query_same)
print("target identical        :", val_target_same)
print("query_batch_id identical:", val_batch_id_same)

if not (
    val_query_same
    and val_target_same
    and val_batch_id_same
):
    raise RuntimeError(
        "Validation sampling is not deterministic."
    )

# ------------------------------------------------------------
# 10. Final status
# ------------------------------------------------------------
print("\n" + "=" * 92)
print("[READY] Original variable-length pipeline restored.")
print(
    "Next step: rebuild and strictly load the frozen "
    "50-step teacher model."
)
print("=" * 92)

Step 3 — Restore original variable-length dataloaders

DataLoader configuration
maximum query points per sample: 8192
batch size                     : 4
num workers                    : 0

Dataset lengths
train: 1600
val  : 200
test : 200

DataLoader lengths
train batches: 400
val batches  : 50
test batches : 50

One train batch
branch          : shape=(4, 1024, 11), dtype=torch.float32
query           : shape=(30078, 2), dtype=torch.float32
target          : shape=(30078, 3), dtype=torch.float32
query_batch_id  : shape=(30078,), dtype=torch.int64
sample_idx      : shape=(4,), dtype=torch.int64
branch_mask     : shape=(4, 1024), dtype=torch.bool
branch_mask valid counts: tensor([1024, 1024, 1024, 1024])

Query points per sample in this batch
tensor([8192, 8192, 5502, 8192])
total concatenated query points: 30078
maximum allowed per sample: 8192

Unique query batch IDs
tensor([0, 1, 2, 3])

Target statistics in normalized space
mean: tensor([-0.2380,  0.0355, -0.0176])
std : tensor([0.9

In [7]:
# ============================================================
# Step 4A — Locate the exact teacher model definition
#
# We do not reconstruct the model yet.
# This cell searches previous notebooks/scripts for:
#   - PointSetDiffusionDenoiser
#   - PointSetBranchNet
#
# It also prints checkpoint state-dict keys and shapes so that
# the next cell can load the teacher strictly.
# ============================================================

from pathlib import Path
import json
import re
import torch

print("=" * 92)
print("Step 4A — Locate exact teacher model implementation")
print("=" * 92)

SEARCH_ROOTS = [
    PROJECT_ROOT / "channel_diffusion_dataset",
    PROJECT_ROOT / "DeepONet",
    PROJECT_ROOT,
]

SEARCH_TERMS = [
    "class PointSetDiffusionDenoiser",
    "class PointSetBranchNet",
    "PointSetDiffusionDenoiser(",
]

# ------------------------------------------------------------
# 1. Search Python files
# ------------------------------------------------------------
python_matches = []

seen_files = set()

for root in SEARCH_ROOTS:
    if not root.exists():
        continue

    for path in root.rglob("*.py"):
        resolved = path.resolve()

        if resolved in seen_files:
            continue

        seen_files.add(resolved)

        try:
            text = path.read_text(
                encoding="utf-8",
                errors="ignore",
            )
        except Exception:
            continue

        matched_terms = [
            term
            for term in SEARCH_TERMS
            if term in text
        ]

        if matched_terms:
            line_matches = []

            lines = text.splitlines()

            for line_number, line in enumerate(
                lines,
                start=1,
            ):
                if any(term in line for term in SEARCH_TERMS):
                    line_matches.append(
                        (line_number, line.strip())
                    )

            python_matches.append(
                {
                    "path": path,
                    "terms": matched_terms,
                    "lines": line_matches,
                }
            )

print("\nPython matches")
print("-" * 92)

if len(python_matches) == 0:
    print("No matching Python source files found.")
else:
    for item in python_matches:
        print(f"\nFile: {item['path']}")

        for line_number, line in item["lines"][:20]:
            print(
                f"  line {line_number:5d}: {line}"
            )

# ------------------------------------------------------------
# 2. Search notebook source cells
# ------------------------------------------------------------
notebook_matches = []

seen_notebooks = set()

for root in SEARCH_ROOTS:
    if not root.exists():
        continue

    for path in root.rglob("*.ipynb"):
        resolved = path.resolve()

        if resolved in seen_notebooks:
            continue

        seen_notebooks.add(resolved)

        try:
            notebook = json.loads(
                path.read_text(
                    encoding="utf-8",
                    errors="ignore",
                )
            )
        except Exception:
            continue

        matching_cells = []

        for cell_index, cell in enumerate(
            notebook.get("cells", [])
        ):
            if cell.get("cell_type") != "code":
                continue

            source = "".join(
                cell.get("source", [])
            )

            matched_terms = [
                term
                for term in SEARCH_TERMS
                if term in source
            ]

            if matched_terms:
                first_nonempty_lines = [
                    line.strip()
                    for line in source.splitlines()
                    if line.strip()
                ][:8]

                matching_cells.append(
                    {
                        "cell_index": cell_index,
                        "terms": matched_terms,
                        "preview": first_nonempty_lines,
                    }
                )

        if matching_cells:
            notebook_matches.append(
                {
                    "path": path,
                    "cells": matching_cells,
                }
            )

print("\nNotebook matches")
print("-" * 92)

if len(notebook_matches) == 0:
    print("No matching notebook cells found.")
else:
    for item in notebook_matches:
        print(f"\nNotebook: {item['path']}")

        for cell in item["cells"]:
            print(
                f"  cell index: {cell['cell_index']}"
            )
            print(
                f"  matched   : {cell['terms']}"
            )

            for line in cell["preview"]:
                print(f"      {line}")

# ------------------------------------------------------------
# 3. Inspect teacher checkpoint state dictionary
# ------------------------------------------------------------
teacher_state_dict = teacher_checkpoint[
    "model_state_dict"
]

print("\nTeacher checkpoint state dictionary")
print("-" * 92)

print("Number of stored tensors:", len(teacher_state_dict))

total_parameter_values = sum(
    tensor.numel()
    for tensor in teacher_state_dict.values()
)

print(
    "Total parameter values:",
    f"{total_parameter_values:,}",
)

print("\nParameter names and shapes")

for key, tensor in teacher_state_dict.items():
    print(
        f"{key:72s} {tuple(tensor.shape)}"
    )

# ------------------------------------------------------------
# 4. Group checkpoint parameters by top-level module
# ------------------------------------------------------------
module_summary = {}

for key, tensor in teacher_state_dict.items():
    top_level_name = key.split(".")[0]

    if top_level_name not in module_summary:
        module_summary[top_level_name] = {
            "n_tensors": 0,
            "n_values": 0,
        }

    module_summary[top_level_name]["n_tensors"] += 1
    module_summary[top_level_name]["n_values"] += tensor.numel()

print("\nTop-level checkpoint modules")
print("-" * 92)

for module_name, info in module_summary.items():
    print(
        f"{module_name:30s} | "
        f"tensors={info['n_tensors']:3d} | "
        f"values={info['n_values']:,}"
    )

# ------------------------------------------------------------
# 5. Check whether the class already exists in this notebook
# ------------------------------------------------------------
class_already_defined = (
    "PointSetDiffusionDenoiser" in globals()
)

print("\nCurrent notebook namespace")
print("-" * 92)
print(
    "PointSetDiffusionDenoiser already defined:",
    class_already_defined,
)

if class_already_defined:
    print(
        "Class object:",
        PointSetDiffusionDenoiser,
    )

# ------------------------------------------------------------
# 6. Print saved configuration
# ------------------------------------------------------------
print("\nSaved model configuration")
print("-" * 92)

for key, value in teacher_model_config.items():
    print(f"{key:32s}: {value}")

print("\nSaved diffusion configuration")
print("-" * 92)

for key, value in teacher_diffusion_config.items():
    print(f"{key:32s}: {value}")

# ------------------------------------------------------------
# 7. Final status
# ------------------------------------------------------------
print("\n" + "=" * 92)

if class_already_defined:
    print(
        "[READY] Model class already exists in the notebook."
    )

elif python_matches or notebook_matches:
    print(
        "[FOUND] Original model definition was located."
    )
    print(
        "Copy the complete model-definition cell from the "
        "matching source into the next notebook cell."
    )

else:
    print(
        "[NOT FOUND] The model definition was not located."
    )
    print(
        "We will reconstruct it from the checkpoint state dict, "
        "but should first inspect all parameter names."
    )

print("=" * 92)

Step 4A — Locate exact teacher model implementation

Python matches
--------------------------------------------------------------------------------------------

File: /home/nuoxu9/PIDIF/DeepONet/fluent_deeponet.py
  line   122: class PointSetBranchNet(nn.Module):

File: /home/nuoxu9/PIDIF/FNO/fluent_deeponet.py
  line   652: class PointSetBranchNet(nn.Module):

Notebook matches
--------------------------------------------------------------------------------------------

Notebook: /home/nuoxu9/PIDIF/train_domain_channel_diffusion_iterative_prediction.ipynb
  cell index: 8
  matched   : ['class PointSetDiffusionDenoiser', 'PointSetDiffusionDenoiser(']
      # ============================================================
      # Step 6: Point-set diffusion denoiser forward check
      # Reuse senior's PointSetBranchNet as branch encoder
      # ============================================================
      import math
      import torch
      import torch.nn as nn
      from fluent_de